In [ ]:
%matplotlib inline
from cpcv_analysis.data import load_data_range
from cpcv_analysis.backtest_engine import (
    slice_by_dates, holdout_sharpe,
    cpcv_vs_holdout_plot, wf_vs_holdout_plot,
)
from cpcv_analysis.config import (
    XGB_PARAMS,
    DEV_START, DEV_END,
    RETRAIN_START, RETRAIN_END,
    HOLDOUT_START, HOLDOUT_END,
)
from xgboost import XGBClassifier

clf = XGBClassifier(**XGB_PARAMS)

In [ ]:
# Carga datos 2023-Q4 → 2025, sin crash sintético
X_all, y_all, t1_all, prices_all, fwd_all = load_data_range(use_crash=False)
print(f'Total obs: {len(X_all)}  |  {X_all.index[0].date()} → {X_all.index[-1].date()}')

In [ ]:
# Particionar por fechas
X_dev,  y_dev,  t1_dev,  fwd_dev  = slice_by_dates(X_all, y_all, t1_all, fwd_all, DEV_START,     DEV_END)
X_wf,   y_wf,   t1_wf,   fwd_wf   = slice_by_dates(X_all, y_all, t1_all, fwd_all, end=DEV_END)
X_ret,  y_ret,  t1_ret,  fwd_ret  = slice_by_dates(X_all, y_all, t1_all, fwd_all, RETRAIN_START, RETRAIN_END)
X_ho,   y_ho,   t1_ho,   fwd_ho   = slice_by_dates(X_all, y_all, t1_all, fwd_all, HOLDOUT_START, HOLDOUT_END)

print(f'Dev set (CPCV): {len(X_dev)} obs  {X_dev.index[0].date()} → {X_dev.index[-1].date()}')
print(f'WF input:       {len(X_wf)}  obs  {X_wf.index[0].date()}  → {X_wf.index[-1].date()}')
print(f'Retrain:        {len(X_ret)} obs  {X_ret.index[0].date()} → {X_ret.index[-1].date()}')
print(f'Hold-out:       {len(X_ho)}  obs  {X_ho.index[0].date()}  → {X_ho.index[-1].date()}')

In [ ]:
# Figura 1: CPCV vs Hold-out
cpcv_sharpes, ho_sr_cpcv = cpcv_vs_holdout_plot(
    clf,
    X_dev, y_dev, t1_dev, fwd_dev,
    X_ret, y_ret,
    X_ho, fwd_ho,
    prices_full=prices_all,
)

In [ ]:
# Figura 2: WalkForward vs Hold-out
wf_sharpes, ho_sr_wf = wf_vs_holdout_plot(
    clf,
    X_wf, y_wf, t1_wf, fwd_wf,
    X_ret, y_ret,
    X_ho, fwd_ho,
    prices_full=prices_all,
    dev_start=DEV_START,
    n_splits=5,
)

In [ ]:
# Tabla resumen
import pandas as pd, numpy as np

summary = pd.DataFrame({
    'CPCV (5 paths)': {
        'n':            len(cpcv_sharpes),
        'mediana':      round(float(np.median(cpcv_sharpes)), 3),
        'media':        round(float(cpcv_sharpes.mean()), 3),
        'std':          round(float(cpcv_sharpes.std()), 3),
        'P5':           round(float(np.percentile(cpcv_sharpes, 5)), 3),
        'P95':          round(float(np.percentile(cpcv_sharpes, 95)), 3),
        'Hold-out SR':  round(ho_sr_cpcv, 3),
        'HO in [P5,P95]': float(np.percentile(cpcv_sharpes, 5)) <= ho_sr_cpcv <= float(np.percentile(cpcv_sharpes, 95)),
    },
    'WalkForward (5 folds)': {
        'n':            len(wf_sharpes),
        'mediana':      round(float(np.median(wf_sharpes)), 3),
        'media':        round(float(wf_sharpes.mean()), 3),
        'std':          round(float(wf_sharpes.std()), 3),
        'P5':           round(float(np.percentile(wf_sharpes, 5)), 3),
        'P95':          round(float(np.percentile(wf_sharpes, 95)), 3),
        'Hold-out SR':  round(ho_sr_wf, 3),
        'HO in [P5,P95]': float(np.percentile(wf_sharpes, 5)) <= ho_sr_wf <= float(np.percentile(wf_sharpes, 95)),
    },
}).T

print('=== Resumen: CPCV vs WalkForward + Hold-out ===')
display(summary)